# Topic 4 - relations

The OMW relation graph the `ConceptRelation` stubs expect. See
[`05.54_data_enrich.md`](../../scratch_space/09_concept_model/05.54_data_enrich/05.54_data_enrich.md) Topic 4.

Open questions: which relation types `wn` exposes and at what coverage;
synset-level vs sense-level; degree distribution and isolated concepts.

## Setup

Thin caller over the staged cache and the OMW wordnets.

In [ ]:
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import wn
from loguru import logger as lg

from lang_tools.lexicon.ingestion.sources.omw import OMW_LEXICONS, OMW_VERSION
from lang_tools.params.lang_tools_params import get_lang_tools_params

LANGS = ["en", "pt", "es", "fr", "it"]
paths = get_lang_tools_params().paths
data_fol = paths.data_fol
staging = data_fol / "_raw/lexicon/staging"
wn.config.data_directory = str(data_fol / "_raw/lexicon/wn_data")


def staged(dataset: str, lang: str) -> pd.DataFrame:
    """Read a staged parquet (``<staging>/<dataset>/<lang>.parquet``)."""
    return pq.read_table(staging / dataset / f"{lang}.parquet").to_pandas()


def wordnet(lang: str) -> wn.Wordnet:
    """Open the pinned OMW lexicon for a language (one wordnet, no merging)."""
    return wn.Wordnet(lexicon=f"{OMW_LEXICONS[lang]}:{OMW_VERSION}")


def ili_of(synset: wn.Synset) -> str | None:
    """Return the synset's ILI id as a plain string, or ``None``."""
    il = synset.ili
    return getattr(il, "id", il) or None


lg.info("staging at {}", staging)

## Synset-level relations + degree

In [ ]:
# Synset-level relation edge counts + degree distribution (en, ILI-keyed).
rel = Counter()
deg = Counter()
n = isolated = 0
for s in wordnet("en").synsets():
    n += 1
    d = 0
    for k, v in s.relations().items():
        rel[k] += len(v)
        d += len(v)
    if d == 0:
        isolated += 1
    deg[d] += 1
print(f"isolated (degree 0): {isolated} / {n} ({100 * isolated / n:.0f}%)")
synset_rel = pd.Series(rel).sort_values(ascending=False)
synset_rel

## Sense-level relations (antonym, derivation)

In [ ]:
# Sense-level relations (en): antonym + derivation live here, not on synsets.
sense_rel = Counter()
for se in wordnet("en").senses():
    for k, v in se.relations().items():
        sense_rel[k] += len(v)
pd.Series(sense_rel).sort_values(ascending=False)

## Findings (measured 2026-06-21)

- **Synset-level graph is rich and ILI-keyed (concept-level).** hypernym /
  hyponym dominate (89,089 edges each), then member/part/substance holonymy and
  meronymy, `similar` (23,134), domain links, `also`, `entails`, `causes`. Only
  6% of synsets are isolated (degree 0).
- **Antonym and derivation are sense-level, not synset-level:** derivation
  74,708 edges, antonym 7,979, pertainym 8,023. They must be read off
  `sense.relations()`, then resolved to the senses' concepts.
- All edges key on ILI, so they resolve directly onto the ILI-grouped concepts
  the current build already produces.

**Decision for Step 4 / phase 7:** capture hypernym + hyponym at minimum from
the synset traversal (cheap, dense), add holonym / meronym / similar while we
are there; capture antonym from the sense traversal for the
`FalseFriendRelation` path. Expose a degree / connectivity metric to the
phase-6 ranking.